# NB03 — Hessen/Darmstadt Inference

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `harish77718/darmstadt-dop20-presliced` — Hessen DOP20 pre-sliced 256×256 PNG patches

**What this does:**
1. Env setup + clone `HarishDeepak/SegEarth-OV-3`
2. Run inference on ~1267 pre-sliced 256×256 patches
   - `cls_hessen.txt` (6 classes, enriched multi-synonym prompts)
   - Each patch runs directly — no sliding window needed (already 256px)
3. Save RGB overlay PNGs for visual comparison

> Visual comparison only for now — no OSM F1 metric.

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned → {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Run Hessen inference

Config: `cls_hessen.txt` (6-class, enriched synonyms).

**No sliding window on pre-sliced patches** — `slide_crop=256` and patches are 256×256,
so `256 < 256` is False and every patch gets a single forward pass (`_inference_single_view`).
SAM3 resizes each 256×256 patch to 1008×1008 internally (`Sam3Processor`, `resolution=1008`).

**`SELECTED_PATCHES`** — edit the list in the cell below to run only specific files.
Leave it empty (`[]`) to run all patches (~40–60 min).

In [ ]:
%%bash
export MPLBACKEND=Agg
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os, time
os.environ["MPLBACKEND"] = "Agg"
torch.manual_seed(42); np.random.seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── USER CONFIG ────────────────────────────────────────────────────────────────
# N_SAMPLE: number of evenly-spaced patches to run. All will be visualized.
#   Set to None to run ALL 1296 patches (~2.2h with 6 queries/class).
#   Default 100 → ~10 minutes.
N_SAMPLE = 100

# SELECTED_PATCHES: override N_SAMPLE with specific filenames (stems or full names).
#   Leave empty [] to use N_SAMPLE.
# Example:
#   SELECTED_PATCHES = ["dop20_32_474_5532_1_he_y0_x0", "dop20_32_474_5532_1_he_y5_x3"]
SELECTED_PATCHES = []
# ───────────────────────────────────────────────────────────────────────────────

INPUT_FOLDER = Path("/kaggle/input/datasets/harish77718/darmstadt-dop20-presliced/darmstadt_dop20/images")
all_paths = sorted(INPUT_FOLDER.glob("*.png"))

if not all_paths:
    print(f"No images found at {INPUT_FOLDER}")
    for p in sorted(INPUT_FOLDER.parent.rglob("*"))[:20]: print(" ", p)
    raise SystemExit(1)

if SELECTED_PATCHES:
    selected_stems = {Path(s).stem for s in SELECTED_PATCHES}
    image_paths = [p for p in all_paths if p.stem in selected_stems]
    missing = selected_stems - {p.stem for p in image_paths}
    if missing:
        print(f"WARNING: {len(missing)} patch(es) not found: {sorted(missing)}")
    vis_indices = set(range(len(image_paths)))
    print(f"Running {len(image_paths)} selected patch(es) — all visualized.")
elif N_SAMPLE is not None:
    step = max(1, len(all_paths) // N_SAMPLE)
    image_paths = all_paths[::step][:N_SAMPLE]
    vis_indices = set(range(len(image_paths)))
    print(f"Running {len(image_paths)} sampled patches (every {step}th of {len(all_paths)}) — all visualized.")
else:
    image_paths = all_paths
    step = max(1, len(image_paths) // 20)
    vis_indices = set(range(0, len(image_paths), step)[:20])
    print(f"Running ALL {len(image_paths)} patches — visualizing {len(vis_indices)} evenly-spaced.")

CLASS_NAMES = ["agricultural field", "forest", "rooftop", "road", "water body", "background"]
COLOR_MAP = np.array([
    [210, 180, 140], [34, 139, 34], [220, 20, 60],
    [105, 105, 105], [30, 144, 255], [200, 200, 200],
], dtype=np.uint8)

model = SegEarthOV3Segmentation(
    type="SegEarthOV3Segmentation",
    classname_path="./configs/cls_hessen.txt",
    prob_thd=0.1, bg_idx=5, confidence_threshold=0.1,
    slide_stride=128, slide_crop=256,
)

VIS_DIR = Path("/kaggle/working/hessen_vis")
VIS_DIR.mkdir(parents=True, exist_ok=True)

for idx, img_path in enumerate(tqdm(image_paths, desc="Hessen inference")):
    t0 = time.time()
    img = Image.open(img_path).convert("RGB")
    ds = SegDataSample()
    ds.set_metainfo({"img_path": str(img_path), "ori_shape": img.size[::-1]})
    img_tensor = transforms.ToTensor()(img).unsqueeze(0).to("cuda")
    result = model.predict(img_tensor, data_samples=[ds])
    seg_pred = result[0].pred_sem_seg.data.cpu().numpy().squeeze(0)

    if idx in vis_indices:
        seg_rgb = COLOR_MAP[np.clip(seg_pred, 0, len(COLOR_MAP) - 1)]
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(img); axes[0].axis("off"); axes[0].set_title("RGB (Hessen DOP20)")
        axes[1].imshow(img); axes[1].imshow(seg_rgb, alpha=0.5)
        axes[1].axis("off"); axes[1].set_title("Prediction — cls_hessen (6 classes)")
        fig.legend(
            handles=[Patch(facecolor=c/255.0, label=n) for n, c in zip(CLASS_NAMES, COLOR_MAP)],
            loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.04)
        )
        plt.tight_layout(rect=[0, 0.07, 1, 1])
        plt.savefig(VIS_DIR / f"{img_path.stem}.png", dpi=100, bbox_inches="tight")
        plt.close()

    elapsed = time.time() - t0
    print(f"  [{idx+1}/{len(image_paths)}] {img_path.name} → {elapsed:.1f}s{'  [saved]' if idx in vis_indices else ''}")

print(f"\nDone. {len(image_paths)} patches processed.")
print(f"Visuals saved to: /kaggle/working/hessen_vis/")
EOF

In [ ]:
import re, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from PIL import Image

SOURCE_IMAGE = "dop20_32_474_5532_1_he"
PATCH_SIZE   = 256

INPUT_FOLDER = Path("/kaggle/input/datasets/harish77718/darmstadt-dop20-presliced/darmstadt_dop20/images")
PRED_DIR     = Path("/kaggle/working/hessen_preds")

CLASS_NAMES = ["agricultural field", "forest", "building", "road", "water body", "background"]
COLOR_MAP = np.array([
    [210, 180, 140], [34, 139, 34], [220, 20, 60],
    [105, 105, 105], [30, 144, 255], [200, 200, 200],
], dtype=np.uint8)

# Collect all pred files for this source image and parse y/x offsets
pred_files = sorted(PRED_DIR.glob(f"{SOURCE_IMAGE}_y*_x*_pred.npy"))
if not pred_files:
    print("No predictions found — run inference cell first.")
else:
    coords = []
    for pf in pred_files:
        m = re.search(r"_y(\d+)_x(\d+)_pred", pf.name)
        if m:
            coords.append((int(m.group(1)), int(m.group(2)), pf))

    max_y = max(y for y, x, _ in coords)
    max_x = max(x for y, x, _ in coords)
    H = max_y + PATCH_SIZE
    W = max_x + PATCH_SIZE

    rgb_canvas  = np.zeros((H, W, 3), dtype=np.uint8)
    pred_canvas = np.zeros((H, W),    dtype=np.uint8)

    for y, x, pf in coords:
        stem = pf.stem.replace("_pred", "")
        img_path = INPUT_FOLDER / f"{stem}.png"
        img = np.array(Image.open(img_path).convert("RGB"))
        pred = np.load(str(pf))
        rgb_canvas [y:y+PATCH_SIZE, x:x+PATCH_SIZE] = img
        pred_canvas[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = pred

    pred_rgb = COLOR_MAP[np.clip(pred_canvas, 0, len(COLOR_MAP)-1)]

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(rgb_canvas);  axes[0].set_title(f"RGB — {SOURCE_IMAGE}", fontsize=13); axes[0].axis("off")
    axes[1].imshow(rgb_canvas);  axes[1].imshow(pred_rgb, alpha=0.55)
    axes[1].set_title("Prediction overlay — cls_hessen (6 classes)", fontsize=13); axes[1].axis("off")
    fig.legend(
        handles=[Patch(facecolor=c/255.0, label=n) for n, c in zip(CLASS_NAMES, COLOR_MAP)],
        loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.01), fontsize=11
    )
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    out = Path(f"/kaggle/working/{SOURCE_IMAGE}_mosaic.png")
    plt.savefig(out, dpi=80, bbox_inches="tight")
    plt.show()
    print(f"Mosaic saved: {out}  ({W}×{H}px from {len(coords)} patches)")

## 4 — OSM pseudo-GT F1 (optional)

Requires OSM masks to be available at `/kaggle/working/osm_masks.npz`.
Generate them first using `segearth_utils.osm_eval.build_osm_masks()`
or download the pre-built file from HF if available.

Always report as **"OSM pseudo-GT F1"** — not ground-truth F1.

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'EOF'
import numpy as np
from pathlib import Path

OSM_MASKS_PATH = Path("/kaggle/working/osm_masks.npz")
PREDS_DIR      = Path("/kaggle/working/hessen_preds")

if not OSM_MASKS_PATH.exists():
    print("OSM masks not found — skipping F1 eval.")
    raise SystemExit(0)

osm_data = np.load(str(OSM_MASKS_PATH))
osm_masks = {k: osm_data[k].astype(np.int64) for k in osm_data.files}

pred_files = sorted(PREDS_DIR.glob("*_pred.npy"))
pred_masks = {p.stem.replace("_pred", ""): np.load(str(p)).astype(np.int64)
              for p in pred_files}

CLASS_NAMES = ["agricultural field", "forest", "building", "road", "water body", "background"]

stems = sorted(set(pred_masks) & set(osm_masks))
print(f"Evaluating {len(stems)} patches...")

n_cls = 6
ignore_id = 255
tp = np.zeros(n_cls); fp = np.zeros(n_cls); fn = np.zeros(n_cls)
for stem in stems:
    pred = pred_masks[stem]
    gt   = osm_masks[stem]
    valid = gt != ignore_id
    for cls in range(n_cls):
        pc = (pred == cls) & valid
        lc = (gt   == cls) & valid
        tp[cls] += (pc & lc).sum()
        fp[cls] += (pc & ~lc).sum()
        fn[cls] += (~pc & lc).sum()

prec = tp / (tp + fp + 1e-6)
rec  = tp / (tp + fn + 1e-6)
f1   = 2 * prec * rec / (prec + rec + 1e-6)

print("\n=== OSM pseudo-GT F1 ===")
print(f"{'Class':<22} {'F1':>6}")
print("-" * 30)
for cls, (name, score) in enumerate(zip(CLASS_NAMES, f1)):
    print(f"{name:<22} {score:>6.4f}")
print("-" * 30)
print(f"{'MEAN':<22} {f1.mean():>6.4f}")
print(f"\nPatches evaluated: {len(stems)}")
EOF